In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from dotenv import load_dotenv
import snowflake.connector

sys.path.append("..")

from src.models.train_model import (
    train_and_compare,
    evaluate_on_test,
    save_artifacts,
)

In [ ]:
load_dotenv("../.env")

conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA", "ANALYTICS"),
)

In [ ]:
train_query = """
SELECT *
FROM analytics.train_set
SAMPLE (5)
LIMIT 300000
"""

val_query = """
SELECT *
FROM analytics.val_set
SAMPLE (10)
LIMIT 100000
"""

test_query = """
SELECT *
FROM analytics.test_set
SAMPLE (10)
LIMIT 100000
"""

In [ ]:
train_df = pd.read_sql(train_query, conn)
val_df = pd.read_sql(val_query, conn)
test_df = pd.read_sql(test_query, conn)

train_df.shape, val_df.shape, test_df.shape

In [ ]:
train_df.columns = train_df.columns.str.lower()
val_df.columns = val_df.columns.str.lower()
test_df.columns = test_df.columns.str.lower()

train_df.head()

In [ ]:
train_df["total_amount"].describe()

In [ ]:
train_df = train_df[
    (train_df["total_amount"] > 0) &
    (train_df["total_amount"] < train_df["total_amount"].quantile(0.99))
]

val_df = val_df[
    (val_df["total_amount"] > 0) &
    (val_df["total_amount"] < val_df["total_amount"].quantile(0.99))
]

test_df = test_df[
    (test_df["total_amount"] > 0) &
    (test_df["total_amount"] < test_df["total_amount"].quantile(0.99))
]

In [ ]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)

train_df.to_parquet("../data/processed/train_sample.parquet", index=False)
val_df.to_parquet("../data/processed/val_sample.parquet", index=False)
test_df.to_parquet("../data/processed/test_sample.parquet", index=False)

In [ ]:
best_model_name, best_pipeline, results = train_and_compare(train_df, val_df)

results

In [ ]:
results_df = (
    pd.DataFrame(results)
    .T
    .sort_values("rmse")
)

results_df

In [ ]:
test_metrics = evaluate_on_test(best_pipeline, test_df)

test_metrics

In [ ]:
save_artifacts(
    best_model_name=best_model_name,
    best_pipeline=best_pipeline,
    results=results,
    test_metrics=test_metrics,
)